In [39]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest
from scipy.stats import spearmanr
file = "data/Medicare_Physician_Other_Practitioners_by_Provider_and_Service_2024.csv"

df = pd.read_csv(file)

C:\Users\alnex\AppData\Local\Temp\ipykernel_12916\149773369.py:8: DtypeWarning: Columns (0: Rndrng_Prvdr_State_FIPS, 1: Rndrng_Prvdr_Zip5) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file)


In [40]:
df["services_per_bene"] = (
    df["Tot_Srvcs"] /
    df["Tot_Benes"].replace(0, np.nan)
)

df["services_per_bene_day"] = (
    df["Tot_Srvcs"] /
    df["Tot_Bene_Day_Srvcs"].replace(0, np.nan)
)

df["charge_to_allowed"] = (
    df["Avg_Sbmtd_Chrg"] /
    df["Avg_Mdcr_Alowd_Amt"].replace(0, np.nan)
)

df["payment_to_allowed"] = (
    df["Avg_Mdcr_Pymt_Amt"] /
    df["Avg_Mdcr_Alowd_Amt"].replace(0, np.nan)
)

df.replace([np.inf, -np.inf], np.nan, inplace=True)


,Rndrng_NPI,Rndrng_Prvdr_Last_Org_Name,Rndrng_Prvdr_First_Name,Rndrng_Prvdr_MI,Rndrng_Prvdr_Crdntls,Rndrng_Prvdr_Ent_Cd,Rndrng_Prvdr_St1,Rndrng_Prvdr_St2,Rndrng_Prvdr_City,Rndrng_Prvdr_State_Abrvtn,...,Tot_Srvcs,Tot_Bene_Day_Srvcs,Avg_Sbmtd_Chrg,Avg_Mdcr_Alowd_Amt,Avg_Mdcr_Pymt_Amt,Avg_Mdcr_Stdzd_Amt,services_per_bene,services_per_bene_day,charge_to_allowed,payment_to_allowed
0,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,36.0,36,284.378889,79.063889,60.828889,62.376389,1.000000,1.0,3.596824,0.769364
1,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,150.0,150,352.176333,126.340800,95.675600,96.007133,1.000000,1.0,2.787511,0.757282
2,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,63.0,63,783.997302,170.360000,134.300476,131.999365,1.000000,1.0,4.602003,0.788333
3,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,16.0,16,122.125000,47.832500,33.350625,33.706250,1.454545,1.0,2.553180,0.697238
4,1003000126,Enkeshafi,Ardalan,NaN,M.D.,I,6410 Rockledge Dr Ste 304,NaN,Bethesda,MD,...,70.0,70,583.606857,81.428571,62.728714,58.486857,1.891892,1.0,7.167102,0.770353
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9781668,1992999825,Deschenes,Geoffrey,R,M.D.,I,1100 9th Ave,Ms:m4-Pfs,Seattle,WA,...,92.0,92,330.000000,140.840326,104.658370,93.407065,1.150000,1.0,2.343079,0.743099
9781669,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,36.0,36,780.583333,165.405000,131.781389,133.834444,1.000000,1.0,4.719225,0.796719
9781670,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,329.0,329,278.127660,74.811337,59.662888,61.002462,1.750000,1.0,3.717721,0.797511
9781671,1992999874,Joffe,Gabriella,NaN,NaN,I,5620 Brook Rd,NaN,Richmond,VA,...,153.0,153,398.522876,113.171438,89.568366,90.994967,1.545455,1.0,3.521409,0.791440


In [41]:
model_features = [
    "Tot_Benes",
    "services_per_bene",
    "services_per_bene_day",
    "charge_to_allowed",
    "payment_to_allowed",
    "Avg_Sbmtd_Chrg"
]

peer_cols = [
    "Rndrng_Prvdr_Type",
    "HCPCS_Cd",
    "Place_Of_Srvc"
]

In [42]:
valid_mask = df[model_features].notna().all(axis=1)

df_valid = df.loc[valid_mask].copy()

In [43]:
print("Original rows:", len(df))
print("Rows with complete model features:", len(df_valid))

Original rows: 9781673
Rows with complete model features: 9781655


In [44]:
peer_summary = (
    df_valid
    .groupby(peer_cols)
    .size()
    .reset_index(name="group_size")
)
peer_summary["size_bucket"] = pd.cut(
    peer_summary["group_size"],
    bins=[0, 29, 99, np.inf],
    labels=[
        "under_30",
        "30_to_99",
        "100_plus"
    ]
)
peer_summary["size_bucket"].value_counts()

size_bucket
under_30    47400
100_plus     6966
30_to_99     5413
Name: count, dtype: int64

In [45]:
def get_peer_group(df_source, provider_type, hcpcs, pos):

    return df_source[
        (df_source["Rndrng_Prvdr_Type"] == provider_type) &
        (df_source["HCPCS_Cd"] == hcpcs) &
        (df_source["Place_Of_Srvc"] == pos)
    ].copy()

In [46]:
reference_peer_groups = peer_summary[
    peer_summary["group_size"] >= 500
].copy()

print("Available reference groups:", len(reference_peer_groups))

Available reference groups: 2717


In [47]:
reference_peer_groups = reference_peer_groups.sample(
    n=min(50, len(reference_peer_groups)),
    random_state=42
)

In [48]:
def train_iforest(train_group, seed=42):

    X_train = np.log1p(
        train_group[model_features].copy()
    )

    model = IsolationForest(
        n_estimators=200,
        max_samples="auto",
        contamination="auto",
        random_state=seed,
        n_jobs=-1
    )

    model.fit(X_train)

    return model

In [49]:
def score_iforest(model, evaluation_group):

    X_eval = np.log1p(
        evaluation_group[model_features].copy()
    )

    return pd.Series(
        -model.score_samples(X_eval),
        index=evaluation_group.index
    )

In [50]:
sample_sizes = [30, 50, 100, 200]

controlled_results = []

for _, row in reference_peer_groups.iterrows():

    group = get_peer_group(
        df_valid,
        row["Rndrng_Prvdr_Type"],
        row["HCPCS_Cd"],
        row["Place_Of_Srvc"]
    )

    # Fixed evaluation set
    evaluation_group = group.copy()

    # Reference model uses the whole peer group
    reference_model = train_iforest(
        group,
        seed=42
    )

    reference_scores = score_iforest(
        reference_model,
        evaluation_group
    )

    for sample_size in sample_sizes:

        if len(group) < sample_size:
            continue

        spearman_runs = []

        for repeat in range(10):

            training_sample = group.sample(
                n=sample_size,
                random_state=repeat
            )

            sample_model = train_iforest(
                training_sample,
                seed=42
            )

            sample_scores = score_iforest(
                sample_model,
                evaluation_group
            )

            rho, _ = spearmanr(
                reference_scores,
                sample_scores
            )

            spearman_runs.append(rho)

        controlled_results.append({
            "provider_type": row["Rndrng_Prvdr_Type"],
            "hcpcs": row["HCPCS_Cd"],
            "pos": row["Place_Of_Srvc"],
            "full_group_size": len(group),
            "training_size": sample_size,
            "mean_reference_spearman": np.mean(spearman_runs),
            "std_reference_spearman": np.std(spearman_runs)
        })

In [51]:
controlled_results = pd.DataFrame(
    controlled_results
)

In [52]:
controlled_summary = (
    controlled_results
    .groupby("training_size")
    .agg(
        avg_spearman=(
            "mean_reference_spearman",
            "mean"
        ),
        median_spearman=(
            "mean_reference_spearman",
            "median"
        ),
        avg_std=(
            "std_reference_spearman",
            "mean"
        ),
        number_of_groups=(
            "full_group_size",
            "count"
        )
    )
)

controlled_summary

,avg_spearman,median_spearman,avg_std,number_of_groups
training_size,,,,
30,0.900894,0.905498,0.034319,50
50,0.932743,0.935699,0.023301,50
100,0.957622,0.959816,0.013925,50
200,0.972110,0.973134,0.007659,50
